# Figure 2 (simplified) — high-χ comparison twist at goal 45

Companion to `figure-1-simplified` (the goal-45 GA-best Figure 1). This renders
the **high-χ comparison twist** (`random_sigma_hash` from `paper-refs.json`) at
the **same goal 45** as Figure 1 (the published Figure 2 used goal 39), so the
two figures form a matched before/after at one goal: the GA-best twist (Fig 1,
low decision information) vs a scrambled high-χ twist (Fig 2, much higher DI).

Layout follows the published Figure 2: a 2×2 panel of "what's labelled" grids
(left) + the goal-conditioned optimal policy with per-state decision information
as the heatmap (right). Uses `env.set_sigma()` so the cached `sigma_inv` is
refreshed — the label panels read `sigma_inv`, and direct `env.sigma[...] =`
leaves it at the constructor-time identity (the 2026-05-23 audit trap).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# gridvis pilot: co-located support + installed packages, no gridFour on path.
NB = Path.cwd()
if str(NB) not in sys.path:
    sys.path.insert(0, str(NB))

from figure_support import (
    RUN_DIR, ENV_ID, SHAPE, DETERMINISM, BETA, THETA,
    MAX_ITERATIONS, MAX_INFO_ITERATIONS, DI_CMAP, DI_VMIN, DI_VMAX, FIGURES_DIR,
    read_sigma_refs,
)
from figure_support import build_env
from gridcore.info.decision_information import DecisionInformation
from gridcore.planning.policy import Policy
from gridcore.planning.state_distribution import UniformStateDistribution
from gridvis.display_twist import plot_twist_labels_2x2
from gridvis import display as display_utils

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

GOAL = 45                       # match Figure 1 (the published Figure 2 used 39)
LABEL_ORDER = ['N', 'E', 'S', 'W']
FIGURE_2 = FIGURES_DIR / 'figure2_random_sigma'

# The comparison (high-χ) twist — the SAME sigma the paper uses, read from the
# canonical refs (written by figure-1-2 / figure-1).
_refs = read_sigma_refs()
RANDOM_SIGMA_HASH = _refs['random_sigma_hash']
print('comparison sigma hash:', RANDOM_SIGMA_HASH,
      '| published goal was', _refs.get('best_random_goal'), '-> now', GOAL)

In [ ]:
def build_sigma_env(goal, sigma):
    """GridRoom with the twist applied for one goal.

    set_sigma() installs sigma AND refreshes the cached inverse env.sigma_inv,
    which the label panels read. Direct env.sigma[...] = leaves sigma_inv at the
    identity (see figure-1-simplified note / 2026-05-23 audit).
    """
    sigma = np.asarray(sigma, dtype=int)
    env = build_env(ENV_ID, shape=SHAPE, goal=goal, determinism=DETERMINISM)
    full = np.tile(np.arange(env.nA, dtype=int), (env.nS, 1))   # identity rows
    avail = np.asarray(env.available_states, dtype=int)
    full[avail] = sigma[avail]
    env.set_sigma(full)
    env.twist_dynamics()
    env.update_dynamics_for_goals(env.goals)
    return env


def solve_goal(env):
    di = DecisionInformation(
        env, UniformStateDistribution(env), theta=float(THETA),
        max_iterations=int(MAX_ITERATIONS), max_info_iterations=int(MAX_INFO_ITERATIONS))
    policy, _, free = di.get_opt_policy_Z_free_vector(beta=float(BETA))
    info = di.get_decision_information_given_policy(policy).astype(float)
    value, _ = Policy.evaluate_policy(env, policy)
    return np.asarray(policy, float), info, np.asarray(free, float), value.astype(float)


def _make_ax(fig, x, y, w, h, fw, fh):
    return fig.add_axes([x / fw, y / fh, w / fw, h / fh])


def _plot_goal_panel(ax, e, p, i, goal, cmap, vmin, vmax):
    """DI heatmap + optimal-policy quiver for one goal."""
    display_utils.plot_quiver(e, p, ax=ax, skip_walls=True)
    im = display_utils.heatmap_imshow(e, i, ax, cmap=cmap, vmin=vmin, vmax=vmax)
    display_utils.add_rectangle_to_grid_for_goals(ax, e)
    display_utils.add_rectangle_to_grid_for_walls(ax, e)
    display_utils.add_outline_for_cells(ax, e)
    display_utils.add_label_to_plot(ax, i.reshape(e.shape), shift=0.35, size=7, env=e)
    ax.set_title(f'Goal {goal}', fontsize=9)
    return im

In [ ]:
# Load the comparison sigma from the run hive (by hash) and solve at goal 45.
_hive = sorted((RUN_DIR / 'hive_sigma').rglob(f'sigma_id={RANDOM_SIGMA_HASH}/sigma.npy'))
if not _hive:
    raise FileNotFoundError(
        f'comparison sigma {RANDOM_SIGMA_HASH} not found under {RUN_DIR}/hive_sigma')
random_sigma = np.load(_hive[0])
env = build_sigma_env(GOAL, random_sigma)
policy, info, _, _ = solve_goal(env)
print(f'comparison σ  goal {GOAL}: mean_info = {float(np.dot(env.isd, info)):.4f}')

In [ ]:
# Figure 2 geometry, identical to the published version (figure-1-2.ipynb): a 2×2
# 'what's labelled' block (left) + a goal panel (right) of the same physical size
# as the Figure 1 goal panel, so the two figures sit side by side cleanly.
pw_a = 2.6; ph_a = 2.6           # goal panel size — shared with Figure 1
rl = 1.1                         # each mini label-panel side (inches)
gy_b = 0.40                      # vertical gap between mini rows (room for titles)
gx_b = 0.12                      # horizontal gap between mini columns
sec_gap = 0.15                   # gap between the 2×2 block and the goal panel
lm = rm = 0.22; tm = 0.38; bm = 0.28
cp = 0.20; cw = 0.12             # reserved right margin (matches published width)

x2_w = 2 * rl + gx_b
x2_h = 2 * rl + gy_b
fw_b = lm + x2_w + sec_gap + pw_a + cp + cw + rm
fh_b = tm + ph_a + bm

fig2 = plt.figure(figsize=(fw_b, fh_b))

def ax_b(x, y, w, h):
    return _make_ax(fig2, x, y, w, h, fw_b, fh_b)

# 2×2 label axes (left), vertically centred on the goal panel
x2_y0 = bm + (ph_a - x2_h) / 2
ax_W = ax_b(lm,              x2_y0 + rl + gy_b, rl, rl)
ax_N = ax_b(lm + rl + gx_b, x2_y0 + rl + gy_b, rl, rl)
ax_E = ax_b(lm,              x2_y0,             rl, rl)
ax_S = ax_b(lm + rl + gx_b, x2_y0,             rl, rl)
plot_twist_labels_2x2(env, policy, axes=np.array([[ax_W, ax_N], [ax_E, ax_S]]),
                      show_titles=True, scale_with_prob=False)

# goal panel (right): optimal policy + per-state DI heatmap at goal 45
goal_ax = ax_b(lm + x2_w + sec_gap, bm, pw_a, ph_a)
_plot_goal_panel(goal_ax, env, policy, info, GOAL, DI_CMAP, DI_VMIN, DI_VMAX)

for ext in ('png', 'pdf'):
    fig2.savefig(f'{FIGURE_2}.{ext}', dpi=150, bbox_inches='tight', pad_inches=0.05)
print(f'saved -> {FIGURE_2}.[png|pdf]')
plt.show()